<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

# Questão 1

Utilize o dataset Iris disponível no scikit-learn.
Divida os dados em treino e teste utilizando divisão estratificada.

**Solução**:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

print("Formato de X_train:", X_train.shape)
print("Formato de X_test:", X_test.shape)
print("Classes no treino:", {classe: int((y_train == classe).sum()) for classe in sorted(set(y_train))})
print("Classes no teste:", {classe: int((y_test == classe).sum()) for classe in sorted(set(y_test))})


# Questão 2

Treine um modelo utilizando `DecisionTreeClassifier`.

Depois calcule:

- acurácia no treino
- acurácia no teste

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train, y_train)

y_train_pred = tree_clf.predict(X_train)
y_test_pred = tree_clf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Acuracia no treino: {train_acc:.4f}")
print(f"Acuracia no teste: {test_acc:.4f}")


# Questão 3

Utilize `plot_tree()` para visualizar a árvore treinada.

Responda:

1. Qual atributo aparece na raiz?
2. Qual é a profundidade da árvore?

**Solução**:

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 10))
plot_tree(
    tree_clf,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
)
plt.show()

root_feature = iris.feature_names[tree_clf.tree_.feature[0]]
tree_depth = tree_clf.get_depth()

print("Atributo na raiz:", root_feature)
print("Profundidade da arvore:", tree_depth)


O atributo que aparece na raiz e `petal length (cm)`.

A profundidade da arvore treinada foi `5`.


# Questão 4

Treine dez árvores com:

- max_depth = 1
- max_depth = 2
- max_depth = 3
...
- max_depth = 9
- max_depth = None

Registre em uma tabela para cada árvore:

- acurácia no treino
- acurácia no teste
- profundidade da árvore
- número de folhas

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depth_values = list(range(1, 10)) + [None]
results = []

print(f"{'max_depth':<10} {'train_acc':<10} {'test_acc':<10} {'depth':<10} {'leaves':<10}")
for depth in depth_values:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score = accuracy_score(y_test, model.predict(X_test))
    tree_depth = model.get_depth()
    leaves = model.get_n_leaves()

    results.append((depth, train_score, test_score, tree_depth, leaves))
    depth_label = str(depth)
    print(f"{depth_label:<10} {train_score:<10.4f} {test_score:<10.4f} {tree_depth:<10} {leaves:<10}")


O overfitting comeca a aparecer a partir de `max_depth = 4`, porque a acuracia de treino continua subindo enquanto a acuracia de teste cai em relacao ao melhor resultado obtido em `max_depth = 3`.

Quando `max_depth=None`, a arvore pode continuar particionando os dados ate memorizar completamente o conjunto de treino. Como o dataset Iris e pequeno e a arvore nao tem restricao de profundidade, ela consegue criar folhas muito especificas e atingir `100%` no treino.


# Questão 5

Treine dois modelos:

- criterion = "gini"
- criterion = "entropy"

Compare:

- profundidade da árvore
- acurácia

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(criterion=criterion, random_state=42)
    model.fit(X_train, y_train)

    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score = accuracy_score(y_test, model.predict(X_test))

    print(f"criterion = {criterion}")
    print(f"  profundidade: {model.get_depth()}")
    print(f"  acuracia no treino: {train_score:.4f}")
    print(f"  acuracia no teste: {test_score:.4f}")
    print()


# Questão 6

Escolha um hiperparâmetro e investigue seu impacto.

Sugestões:

- max_depth
- min_samples_split
- min_samples_leaf
- criterion

Mostre resultados e interprete.
- melhor modelo encontrado
- acurácia
- parâmetros

**Solução**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_result = None

print(f"{'min_leaf':<10} {'train_acc':<10} {'test_acc':<10} {'depth':<10} {'leaves':<10}")
for min_leaf in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    model = DecisionTreeClassifier(min_samples_leaf=min_leaf, random_state=42)
    model.fit(X_train, y_train)

    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score = accuracy_score(y_test, model.predict(X_test))
    depth = model.get_depth()
    leaves = model.get_n_leaves()

    print(f"{min_leaf:<10} {train_score:<10.4f} {test_score:<10.4f} {depth:<10} {leaves:<10}")

    candidate = (test_score, train_score, -depth, min_leaf)
    if best_result is None or candidate > best_result[0]:
        best_result = (
            candidate,
            {
                "min_samples_leaf": min_leaf,
                "train_acc": train_score,
                "test_acc": test_score,
                "depth": depth,
                "leaves": leaves,
            },
        )

print("\nMelhor modelo encontrado:")
print(best_result[1])


Ao investigar `min_samples_leaf`, o melhor resultado de teste apareceu com `min_samples_leaf = 1`, que corresponde ao modelo sem restricao adicional nas folhas.

Melhor modelo encontrado:

- parametro analisado: `min_samples_leaf`
- melhor valor: `1`
- acuracia no treino: `1.0000`
- acuracia no teste: `0.9333`
- profundidade: `5`
- numero de folhas: `8`

Interpretacao: aumentar `min_samples_leaf` simplifica a arvore e reduz a variancia, mas neste experimento isso nao melhorou a generalizacao no conjunto de teste.
